In [ ]:
import numpy as np
from pathlib import Path

import rtde_control 
import rtde_receive 

In [ ]:
ROBOT_IP = "192.168.1.102"
rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)
rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)

# --- READ joint positions (radians, 6 joints) ---
joint_q = rtde_r.getActualQ()
print("Joint positions [rad]:", joint_q)

# --- READ Tool Center Point (TCP) pose ---
# Returns [x, y, z, rx, ry, rz] in meters and axis-angle rotation
tcp_pose = rtde_r.getActualTCPPose()
print("TCP pose [m, rad]:", tcp_pose)

In [ ]:
# Load csv file with x, y, z coordinates
root_dir = Path.cwd().parent
print("Root directory:", root_dir)
csv_path = root_dir / "task2" / "interpolated_rigid_bodies.csv"
waypoints = np.loadtxt(csv_path, delimiter=",", skiprows=1)[350:]
print("Waypoints shape:", waypoints.shape)

In [ ]:
#Scale waypoints to fit within robot workspace

# Apply rotation to all waypoints
# Rotate 90° around X-axis  (Y → Z, Z → -Y)
R_x = np.array([
    [1,  0,  0],
    [0,  0, -1],
    [0,  1,  0]
])

# Rotate 90° around Y-axis  (Z → X, X → -Z)
R_y = np.array([
    [0,  0,  1],
    [0,  1,  0],
    [-1, 0,  0]
])

# Rotate 90° around Z-axis  (X → Y, Y → -X)
R_z = np.array([
    [0, -1,  0],
    [1,  0,  0],
    [0,  0,  1]
])

waypoints = np.dot(waypoints, R_z.T)

waypoints *= 0.15

# Read TCP pose
tcp_pose = rtde_r.getActualTCPPose()

T0 = waypoints[0]

tcp_pos = np.array(tcp_pose[:3])  # Extract x, y, z from TCP pose
tcp_orientation = tcp_pose[3:]  # Extract orientation (rx, ry, rz)

translation_vector = tcp_pos - T0

print("translation vector: ", translation_vector)

# Apply translation to all waypoints
translated_waypoints = waypoints + translation_vector



In [ ]:
# Plot the waypoints and TCP position in 2D (X-Y plane)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.plot(translated_waypoints[:, 1], translated_waypoints[:, 2], 'bo-', label='translated_waypoints')
plt.plot(tcp_pos[1], tcp_pos[2], 'ro', label='TCP Position')
plt.xlabel('Y (m)')
plt.ylabel('Z (m)')
plt.title('translated_waypoints and TCP Position in Y-Z Plane')
plt.legend()
plt.grid()
plt.axis('equal')
plt.show()

In [ ]:
# Plot the waypoints and TCP position
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(waypoints[:, 0], waypoints[:, 1], waypoints[:, 2], label='Waypoints')
ax.scatter(tcp_pos[0], tcp_pos[1], tcp_pos[2], color='red', label='TCP Position')
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
ax.set_zlabel('Z [m]')
ax.set_title('Waypoints and TCP Position')
ax.legend()
plt.show()

In [ ]:
# ── 5. Replay trajectory with moveL path + blending ──────────────────────────
speed        = 0.5    # m/s
acceleration = 0.1    # m/s²
blend        = 0.005   # metres — smooth blending between waypoints (0 = stop at each)



##Move to initial pos
rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)
rtde_c.moveL(translated_waypoints[0], speed,acceleration, blend)

In [ ]:
path = []
print("TCP orientation (rx, ry, rz):", tcp_orientation)
for i, point in enumerate(translated_waypoints):
    # Last waypoint must have blend = 0 (no blending at the end)
    b = 0.0 if i == len(translated_waypoints) - 1 else blend
    point = list(point) + tcp_orientation + [speed, acceleration, b]
    path.append(point)

rtde_c.moveL(path)   # executes the full path in one call with smooth blending

# ── 6. Clean up ──────────────────────────────────────────────────────────────
rtde_c.stopScript()
rtde_c.disconnect()
rtde_r.disconnect()